# Данные L2

Потоковые ряды и итоговые флаги контроля качества читаются из отдельных
`*_fluxes.csv` уровня L1. При экспорте L2 сохраняет флаги L1. Для ERA5
сохраняется прежний расчёт `Cd`.

In [ ]:
from flux_qc import FLAG_COLUMNS
from l0_data import MonthlyWriter, REFERENCE_ROOT, read_city, series_directory

city_name = 'Moscow'
df4sites = read_city(city_name, level='L1', include_reference=True)
for site, series in df4sites.items():
    if site == 'ERA5':
        df4sites[site] = series.assign(Cd=(series['Ustar'] / series['Wind']) ** 2)
    else:
        missing = set(FLAG_COLUMNS) - set(series.columns)
        if missing:
            raise ValueError(f'{site} — в L1 нет флагов {sorted(missing)}. Подготовьте данные в process_L1.ipynb.')

display(list(df4sites))


In [ ]:
if city_name == 'Tomsk':
    import pandas as pd
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    vars2draw = ['Qh', 'Qtau', 'Cd']
    site2draw = 'TFN-Oblkom-30m'


    df2draw = pd.DataFrame(df4sites[site2draw][vars2draw])
    df2draw_f = df2draw.where(df4sites[site2draw]['qc_tot'] > 0)

    # df2draw = df2draw.resample('1D').mean()
    # df2draw_f = df2draw_f.resample('1D').mean()


    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04, 
    )

    fig.add_trace(
        go.Scatter(x=df2draw.index, y=df2draw['Qh'], name='Qh (raw)', mode='lines',  line=dict(width=2)), #, line=dict(color='blue')),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(x=df2draw_f.index, y=df2draw_f['Qh'], name='Qh (qc)', mode='lines',  line=dict(width=2)), #, line=dict(color='red')),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(x=df2draw.index, y=df2draw['Qtau'], name='Qtau (raw)', mode='lines',  line=dict(width=2)), #, line=dict(color='blue')),
        row=2, col=1
    )

    fig.add_trace(
        go.Scatter(x=df2draw_f.index, y=df2draw_f['Qtau'], name='Qtau (qc)', mode='lines',  line=dict(width=2)), #, line=dict(color='red')),
        row=2, col=1
    )

    fig.add_trace(
        go.Scatter(x=df2draw.index, y=df2draw['Cd'], name='Cd (raw)', mode='lines',  line=dict(width=2)), #, line=dict(color='blue')),
        row=3, col=1
    )

    fig.add_trace(
        go.Scatter(x=df2draw_f.index, y=df2draw_f['Cd'], name='Cd (qc)', mode='lines',  line=dict(width=2)), #, line=dict(color='red')),
        row=3, col=1
    )

    fig.update_xaxes(range=['2025-04-01', '2025-05-01'])
    fig.update_yaxes(title_text='Qh (W/m²)', range=[-200, 500], row=1, col=1)
    fig.update_yaxes(title_text='Qtau (N/m²)', range=[0, 4], row=2, col=1)
    fig.update_yaxes(title_text='Cd', range=[0, 0.2], row=3, col=1)

    #fig.update_xaxes(range=['2024-06-01', '2025-07-01'])
    # fig.update_yaxes(title_text='Qh (W/m²)', range=[-50, 200], row=1, col=1)
    # fig.update_yaxes(title_text='Qtau (N/m²)', range=[0, 1], row=2, col=1)
    # fig.update_yaxes(title_text='Cd', range=[0, 0.1], row=3, col=1)


    fig.update_layout(
        width=800, height = 600,
        margin=dict(l=10, r=10, t=10, b=10),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.1,
            )
    )
    fig.show()


    from l0_data import REPORTS_ROOT, RELEASE
    report_dir = REPORTS_ROOT / 'L2' / city_name / RELEASE
    report_dir.mkdir(parents=True, exist_ok=True)
    fig.write_image(str(report_dir / f'{site2draw}.qc.png'), scale=4)


In [ ]:
for site, series in df4sites.items():
    if site == 'ERA5':
        directory = REFERENCE_ROOT / 'L2' / city_name
    else:
        directory = series_directory(city_name, 'fluxes', site, level='L2')
    writer = MonthlyWriter(directory)
    writer.write(series)
    writer.finish()
